In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Nehru_Nagar_Delhi_DPCC_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,377.0,182.0,252.0,116.0,176.0,193.0,87.0,67.0,83.0,124.0,356.0,323.0
1,2,384.0,289.0,112.0,110.0,172.0,235.0,107.0,65.0,73.0,143.0,NaN,335.0
2,3,393.0,257.0,116.0,178.0,232.0,259.0,96.0,62.0,74.0,208.0,430.0,291.0
3,4,427.0,331.0,228.0,160.0,279.0,238.0,66.0,62.0,61.0,175.0,398.0,252.0
4,5,389.0,240.0,185.0,146.0,288.0,228.0,68.0,63.0,167.0,139.0,403.0,232.0
5,6,365.0,205.0,210.0,137.0,226.0,157.0,66.0,64.0,85.0,132.0,367.0,238.0
6,7,380.0,221.0,178.0,135.0,279.0,249.0,61.0,64.0,58.0,124.0,406.0,279.0
7,8,401.0,203.0,170.0,142.0,225.0,255.0,61.0,63.0,120.0,185.0,405.0,371.0
8,9,409.0,142.0,179.0,168.0,257.0,222.0,80.0,64.0,138.0,163.0,380.0,208.0
9,10,330.0,349.0,183.0,268.0,168.0,203.0,138.0,65.0,102.0,123.0,359.0,296.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [6]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [7]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [8]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,377.000000,182.000000,252.0,116.000000,176.0,193.000000,87.000000,67.000000,83.000000,124.000000,356.000000,323.0
1,2,384.000000,289.000000,112.0,110.000000,172.0,235.000000,107.000000,65.000000,73.000000,143.000000,355.848485,335.0
2,3,393.000000,257.000000,116.0,178.000000,232.0,259.000000,96.000000,62.000000,74.000000,208.000000,430.000000,291.0
3,4,427.000000,331.000000,228.0,160.000000,279.0,238.000000,66.000000,62.000000,61.000000,175.000000,398.000000,252.0
4,5,389.000000,240.000000,185.0,146.000000,288.0,228.000000,68.000000,63.000000,167.000000,139.000000,403.000000,232.0
5,6,365.000000,205.000000,210.0,137.000000,226.0,157.000000,66.000000,64.000000,85.000000,132.000000,367.000000,238.0
6,7,380.000000,221.000000,178.0,135.000000,279.0,249.000000,61.000000,64.000000,58.000000,124.000000,406.000000,279.0
7,8,401.000000,203.000000,170.0,142.000000,225.0,255.000000,61.000000,63.000000,120.000000,185.000000,405.000000,371.0
8,9,409.000000,142.000000,179.0,168.000000,257.0,222.000000,80.000000,64.000000,138.000000,163.000000,380.000000,208.0
9,10,330.000000,349.000000,183.0,150.848485,168.0,203.000000,88.088235,65.000000,102.000000,123.000000,359.000000,296.0
